# Fabric Data Agent - Factory

This notebook automatically scans all workspaces in a Fabric tenant, profiles data items
(Lakehouses, Warehouses, Semantic Models, KQL Databases), uses Azure OpenAI to cluster
them into thematic domains, and creates fully configured Data Agents with system instructions,
data instructions, and example queries.

**Phases:**
1. Workspace & Item Discovery
2. Schema & Data Profiling
3. LLM-based Topic Clustering
4. Data Agent Creation & Configuration
5. Reporting & Agent Garden Catalog

## Cell 0 — Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values before running
# ============================================================

# Azure OpenAI settings (for topic clustering)
AOAI_ENDPOINT = "https://<your-resource>.openai.azure.com/"  # Your Azure OpenAI endpoint
AOAI_DEPLOYMENT = "gpt-4o"                                    # Deployment name
AOAI_API_VERSION = "2024-12-01-preview"                       # API version

# Profiling settings
SAMPLE_ROWS = 5              # Number of sample rows to read per table
MAX_TABLES_PER_ITEM = 50     # Skip items with more tables than this (very large schemas)
MAX_COLUMNS_IN_SAMPLE = 20   # Limit columns sent to LLM to control token usage

# Agent creation settings
AGENT_NAME_PREFIX = "Auto"                          # Prefix for auto-created agent names
AGENT_TARGET_WORKSPACE = None                       # Workspace name to create agents in (None = current notebook workspace)
MAX_DATASOURCES_PER_AGENT = 5                       # SDK limit
MAX_EXAMPLES_PER_DATASOURCE = 20                    # How many example queries to generate (max 100)

# Discovery settings
WORKSPACE_FILTER = None        # Set to a list of workspace names to limit scanning, e.g. ["Sales WS", "Finance WS"]. None = all.
ITEM_TYPES_TO_SCAN = ["Lakehouse", "Warehouse", "SemanticModel", "KQLDatabase"]

# Catalog output
CATALOG_LAKEHOUSE = None       # Lakehouse name to write the agent catalog to (None = skip catalog write)
CATALOG_TABLE_NAME = "agent_garden_catalog"

# Safety
DRY_RUN = True                 # If True, only plan agents but don't create them. Set to False to actually create.

print("Configuration loaded.")
print(f"  DRY_RUN = {DRY_RUN} — {'Will NOT create agents (preview only)' if DRY_RUN else 'Will CREATE and PUBLISH agents'}")

## Cell 1 — Install Dependencies

In [ ]:
%pip install fabric-data-agent-sdk openai --quiet

In [ ]:
import json
import time
import re
from collections import defaultdict
from typing import Any

import pandas as pd
import sempy.fabric as fabric
from openai import AzureOpenAI
from synapse.ml.mlflow import get_mlflow_env_config

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

print("All imports successful.")

---
## Phase 1 — Workspace & Item Discovery

Enumerates all accessible workspaces and lists data items (Lakehouses, Warehouses, Semantic Models, KQL Databases).

In [ ]:
def discover_items(workspace_filter: list[str] | None, item_types: list[str]) -> pd.DataFrame:
    """Discover all data items across accessible workspaces."""
    
    # List all workspaces the current user can access
    workspaces_df = fabric.list_workspaces()
    print(f"Found {len(workspaces_df)} accessible workspaces.")
    
    if workspace_filter:
        workspaces_df = workspaces_df[workspaces_df["Name"].isin(workspace_filter)]
        print(f"Filtered to {len(workspaces_df)} workspaces matching filter.")
    
    all_items = []
    
    for _, ws in workspaces_df.iterrows():
        ws_id = ws["Id"]
        ws_name = ws["Name"]
        
        try:
            items_df = fabric.list_items(workspace=ws_id)
        except Exception as e:
            print(f"  [WARN] Cannot list items in '{ws_name}': {e}")
            continue
        
        # Filter for data item types
        data_items = items_df[items_df["Type"].isin(item_types)]
        
        for _, item in data_items.iterrows():
            all_items.append({
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "item_id": item["Id"],
                "item_name": item["Display Name"],
                "item_type": item["Type"],
            })
        
        if len(data_items) > 0:
            print(f"  [{ws_name}] Found {len(data_items)} data items")
    
    inventory_df = pd.DataFrame(all_items)
    print(f"\nTotal: {len(inventory_df)} data items across {inventory_df['workspace_name'].nunique()} workspaces.")
    return inventory_df


# Run discovery
item_inventory_df = discover_items(WORKSPACE_FILTER, ITEM_TYPES_TO_SCAN)
display(item_inventory_df)

---
## Phase 2 — Schema & Data Profiling

For each discovered data item, reads the schema (tables + columns) and samples a few rows to understand the data context.

In [ ]:
# ----- Profiling helpers per item type -----

def profile_lakehouse_or_warehouse(item_name: str, workspace_id: str, sample_rows: int, max_tables: int, max_cols: int) -> list[dict]:
    """Profile a Lakehouse or Warehouse using Spark SQL via the SQL Analytics Endpoint."""
    tables_info = []
    
    try:
        # Use sempy to list tables
        tables_df = fabric.list_tables(workspace=workspace_id, item=item_name)
    except Exception:
        # Fallback: try Spark SQL if sempy doesn't support this item
        try:
            tables_result = spark.sql(f"SHOW TABLES IN `{item_name}`")
            table_names = [row["tableName"] for row in tables_result.collect()]
            tables_df = pd.DataFrame({"Name": table_names})
        except Exception as e:
            print(f"    [WARN] Cannot list tables for '{item_name}': {e}")
            return tables_info
    
    if len(tables_df) > max_tables:
        print(f"    [SKIP] '{item_name}' has {len(tables_df)} tables (>{max_tables}), skipping detail profiling.")
        return [{"name": row.get("Name", str(row)), "columns": [], "sample_values": {}} for _, row in tables_df.iterrows()]
    
    for _, table_row in tables_df.iterrows():
        table_name = table_row.get("Name", str(table_row))
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        
        try:
            # Read schema via Spark
            sample_df = spark.sql(f"SELECT * FROM `{item_name}`.`{table_name}` LIMIT {sample_rows}")
            
            # Columns (limit to max_cols)
            for field in sample_df.schema.fields[:max_cols]:
                table_info["columns"].append(f"{field.name} ({field.dataType.simpleString()})")
            
            # Sample values
            pdf = sample_df.toPandas()
            for col in pdf.columns[:max_cols]:
                vals = pdf[col].dropna().head(sample_rows).tolist()
                # Convert to string for JSON safety
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile table '{item_name}.{table_name}': {e}")
        
        tables_info.append(table_info)
    
    return tables_info


def profile_semantic_model(item_name: str, workspace_id: str, sample_rows: int, max_cols: int) -> list[dict]:
    """Profile a Power BI Semantic Model using sempy."""
    tables_info = []
    
    try:
        tables_df = fabric.list_tables(dataset=item_name, workspace=workspace_id)
    except Exception as e:
        print(f"    [WARN] Cannot list tables for semantic model '{item_name}': {e}")
        return tables_info
    
    for _, table_row in tables_df.iterrows():
        table_name = table_row["Name"]
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        
        try:
            # Get columns
            cols_df = fabric.list_columns(dataset=item_name, workspace=workspace_id, table=table_name)
            for _, col_row in cols_df.iterrows():
                col_name = col_row["Column Name"]
                col_type = col_row.get("Data Type", "unknown")
                table_info["columns"].append(f"{col_name} ({col_type})")
            
            # Sample via DAX TOPN
            safe_columns = [c.split(" (")[0] for c in table_info["columns"][:max_cols]]
            col_refs = ", ".join([f"'{table_name}'[{c}]" for c in safe_columns])
            dax = f"EVALUATE TOPN({sample_rows}, SELECTCOLUMNS('{table_name}', {col_refs}))"
            result_df = fabric.evaluate_dax(dataset=item_name, dax_string=dax, workspace=workspace_id)
            
            for col in result_df.columns[:max_cols]:
                vals = result_df[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile semantic model table '{item_name}.{table_name}': {e}")
        
        tables_info.append(table_info)
    
    return tables_info


def profile_kql_database(item_name: str, workspace_id: str, sample_rows: int, max_cols: int) -> list[dict]:
    """Profile a KQL Database by listing tables and sampling rows."""
    tables_info = []
    
    try:
        # Use Fabric REST to get KQL connection info, then query via kusto client
        # For Fabric-native KQL, we can use the Spark KQL connector
        tables_result = spark.sql(f"SHOW TABLES IN `{item_name}`")
        table_names = [row["tableName"] for row in tables_result.collect()]
    except Exception as e:
        print(f"    [WARN] Cannot list KQL tables for '{item_name}': {e}")
        return tables_info
    
    for table_name in table_names:
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        
        try:
            sample_df = spark.sql(f"SELECT * FROM `{item_name}`.`{table_name}` LIMIT {sample_rows}")
            for field in sample_df.schema.fields[:max_cols]:
                table_info["columns"].append(f"{field.name} ({field.dataType.simpleString()})")
            
            pdf = sample_df.toPandas()
            for col in pdf.columns[:max_cols]:
                vals = pdf[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile KQL table '{item_name}.{table_name}': {e}")
        
        tables_info.append(table_info)
    
    return tables_info

In [ ]:
def profile_all_items(inventory_df: pd.DataFrame) -> list[dict]:
    """Profile all discovered items and return structured profiles."""
    profiled_items = []
    
    for idx, row in inventory_df.iterrows():
        item_name = row["item_name"]
        item_type = row["item_type"]
        ws_id = row["workspace_id"]
        ws_name = row["workspace_name"]
        
        print(f"\n[{idx+1}/{len(inventory_df)}] Profiling {item_type} '{item_name}' in '{ws_name}'...")
        
        if item_type in ("Lakehouse", "Warehouse"):
            tables = profile_lakehouse_or_warehouse(item_name, ws_id, SAMPLE_ROWS, MAX_TABLES_PER_ITEM, MAX_COLUMNS_IN_SAMPLE)
        elif item_type == "SemanticModel":
            tables = profile_semantic_model(item_name, ws_id, SAMPLE_ROWS, MAX_COLUMNS_IN_SAMPLE)
        elif item_type == "KQLDatabase":
            tables = profile_kql_database(item_name, ws_id, SAMPLE_ROWS, MAX_COLUMNS_IN_SAMPLE)
        else:
            print(f"    [SKIP] Unsupported type '{item_type}'")
            continue
        
        profile = {
            "item_id": row["item_id"],
            "item_name": item_name,
            "item_type": item_type,
            "workspace_id": ws_id,
            "workspace_name": ws_name,
            "tables": tables,
            "table_count": len(tables),
        }
        profiled_items.append(profile)
        print(f"    Found {len(tables)} tables.")
    
    print(f"\nProfiling complete: {len(profiled_items)} items profiled.")
    return profiled_items


# Run profiling
profiled_items = profile_all_items(item_inventory_df)

# Summary
summary = pd.DataFrame([{
    "item_name": p["item_name"],
    "item_type": p["item_type"],
    "workspace": p["workspace_name"],
    "tables": p["table_count"],
} for p in profiled_items])
display(summary)

---
## Phase 3 — LLM-based Topic Clustering

Sends the profiled schema + sample data to Azure OpenAI to intelligently cluster data items into thematic Data Agents.

In [ ]:
def build_clustering_prompt(profiled_items: list[dict], max_ds_per_agent: int, max_examples: int) -> str:
    """Build the LLM prompt with all profiled item schemas and sample data."""
    
    # Build a compact representation of each item
    items_description = []
    for p in profiled_items:
        item_desc = f"\n### Data Item: {p['item_name']} (Type: {p['item_type']}, Workspace: {p['workspace_name']})\n"
        for t in p["tables"]:
            item_desc += f"  Table: {t['name']}\n"
            if t["columns"]:
                item_desc += f"    Columns: {', '.join(t['columns'][:15])}\n"
            if t["sample_values"]:
                # Show just a few sample values per column to save tokens
                for col, vals in list(t["sample_values"].items())[:8]:
                    item_desc += f"    Sample {col}: {vals[:3]}\n"
        items_description.append(item_desc)
    
    all_items_text = "\n".join(items_description)
    
    prompt = f"""You are an expert data architect. Your task is to analyze the following data items from a Microsoft Fabric tenant and group them into thematic Data Agents.

RULES:
1. Each Data Agent should cover a coherent business domain (e.g., "Fraud Detection", "Travel Analytics", "Sales Performance").
2. Related topics MUST be merged into the same agent (e.g., Fraud data + Payment data → Fraud & Payment Agent).
3. Each agent can have at most {max_ds_per_agent} data sources. If a theme requires more, split into sub-agents.
4. For each agent, provide:
   - A clear, descriptive name
   - A description (2-3 sentences) explaining what the agent covers
   - Which data items (by name) and which tables from each item belong to this agent
   - System instructions: detailed guidance for the AI on how to answer questions about this domain
   - Per-datasource instructions: specific notes about each data source's tables and relationships
   - Example queries: up to {max_examples} question/SQL pairs per data source (only for Lakehouse, Warehouse, KQLDatabase — NOT for SemanticModel)
5. Every data item MUST be assigned to at least one agent. Don't leave items unassigned.
6. The system instructions should be specific to the data. Reference actual table and column names.
7. Example queries should demonstrate common analytical questions users would ask about this domain.

DATA ITEMS TO ANALYZE:
{all_items_text}

Respond ONLY with valid JSON matching this exact schema:
{{
  "agents": [
    {{
      "name": "Agent Name",
      "description": "What this agent does and what data it covers.",
      "data_sources": [
        {{
          "item_name": "exact item name from above",
          "item_type": "Lakehouse|Warehouse|SemanticModel|KQLDatabase",
          "workspace_name": "exact workspace name",
          "selected_tables": ["table1", "table2"],
          "data_instructions": "Per-datasource instructions for this source.",
          "example_queries": [
            {{"question": "What are the top 10...?", "sql": "SELECT TOP 10..."}}
          ]
        }}
      ],
      "system_instructions": "Detailed system instructions for the agent..."
    }}
  ]
}}
"""
    return prompt


def call_azure_openai(prompt: str) -> dict:
    """Call Azure OpenAI and parse the JSON response."""
    configs = get_mlflow_env_config()
    
    client = AzureOpenAI(
        azure_endpoint=AOAI_ENDPOINT,
        api_version=AOAI_API_VERSION,
        azure_ad_token=configs.driver_aad_token,
    )
    
    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a data architecture expert. Respond only with valid JSON."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        max_tokens=16000,
        response_format={"type": "json_object"},
    )
    
    raw = response.choices[0].message.content
    return json.loads(raw)


def validate_agent_plan(plan: dict, profiled_items: list[dict]) -> list[str]:
    """Validate the LLM-generated agent plan. Returns list of warnings."""
    warnings = []
    known_items = {p["item_name"] for p in profiled_items}
    assigned_items = set()
    
    for agent in plan.get("agents", []):
        name = agent.get("name", "<unnamed>")
        sources = agent.get("data_sources", [])
        
        if len(sources) > MAX_DATASOURCES_PER_AGENT:
            warnings.append(f"Agent '{name}' has {len(sources)} data sources (max {MAX_DATASOURCES_PER_AGENT}). Will be split.")
        
        for ds in sources:
            ds_name = ds.get("item_name", "")
            assigned_items.add(ds_name)
            if ds_name not in known_items:
                warnings.append(f"Agent '{name}' references unknown item '{ds_name}'.")
            
            # Semantic Models don't support example queries
            if ds.get("item_type") == "SemanticModel" and ds.get("example_queries"):
                ds["example_queries"] = []  # Auto-fix
                warnings.append(f"Agent '{name}', source '{ds_name}': Removed example queries (not supported for Semantic Models).")
    
    unassigned = known_items - assigned_items
    if unassigned:
        warnings.append(f"Unassigned items: {unassigned}")
    
    return warnings


def split_oversized_agents(plan: dict) -> dict:
    """Split agents that have more than MAX_DATASOURCES_PER_AGENT sources."""
    new_agents = []
    
    for agent in plan.get("agents", []):
        sources = agent.get("data_sources", [])
        if len(sources) <= MAX_DATASOURCES_PER_AGENT:
            new_agents.append(agent)
        else:
            # Split into chunks
            for i in range(0, len(sources), MAX_DATASOURCES_PER_AGENT):
                chunk = sources[i:i + MAX_DATASOURCES_PER_AGENT]
                part_num = (i // MAX_DATASOURCES_PER_AGENT) + 1
                split_agent = {
                    **agent,
                    "name": f"{agent['name']} Part {part_num}",
                    "data_sources": chunk,
                }
                new_agents.append(split_agent)
    
    plan["agents"] = new_agents
    return plan

In [ ]:
# Build prompt and call LLM
print("Building clustering prompt...")
prompt = build_clustering_prompt(profiled_items, MAX_DATASOURCES_PER_AGENT, MAX_EXAMPLES_PER_DATASOURCE)
print(f"Prompt length: {len(prompt):,} characters")

print("\nCalling Azure OpenAI for topic clustering...")
agent_plan = call_azure_openai(prompt)

print(f"\nLLM proposed {len(agent_plan.get('agents', []))} agents.")

# Validate
warnings = validate_agent_plan(agent_plan, profiled_items)
if warnings:
    print("\nValidation warnings:")
    for w in warnings:
        print(f"  ⚠ {w}")

# Split oversized agents
agent_plan = split_oversized_agents(agent_plan)
print(f"\nFinal plan: {len(agent_plan['agents'])} agents after validation.")

---
## Human Review Checkpoint

Review the proposed agents below. After reviewing, set `DRY_RUN = False` in Cell 0 and re-run Phase 4 to create the agents.

In [ ]:
def display_agent_plan(plan: dict):
    """Display the proposed agent plan in a human-readable format."""
    agents = plan.get("agents", [])
    
    for i, agent in enumerate(agents, 1):
        print(f"{'='*80}")
        print(f"AGENT {i}: {agent['name']}")
        print(f"{'='*80}")
        print(f"Description: {agent.get('description', 'N/A')}")
        print(f"\nSystem Instructions (preview):")
        instructions = agent.get('system_instructions', 'N/A')
        print(f"  {instructions[:500]}{'...' if len(instructions) > 500 else ''}")
        print(f"\nData Sources ({len(agent.get('data_sources', []))}):\n")
        
        for ds in agent.get("data_sources", []):
            print(f"  📦 {ds['item_name']} ({ds['item_type']}) — Workspace: {ds.get('workspace_name', 'N/A')}")
            print(f"     Tables: {', '.join(ds.get('selected_tables', []))}")
            di = ds.get('data_instructions', '')
            if di:
                print(f"     Data Instructions: {di[:200]}{'...' if len(di) > 200 else ''}")
            examples = ds.get('example_queries', [])
            if examples:
                print(f"     Example Queries ({len(examples)}):")
                for eq in examples[:3]:  # Show first 3
                    print(f"       Q: {eq.get('question', '')}")
                    print(f"       A: {eq.get('sql', '')[:120]}")
                if len(examples) > 3:
                    print(f"       ... and {len(examples) - 3} more")
            print()
    
    print(f"\n{'='*80}")
    print(f"TOTAL: {len(agents)} agents planned.")
    if DRY_RUN:
        print("\n👉 DRY_RUN is ON. Set DRY_RUN = False in Cell 0 and re-run Phase 4 to create agents.")
    else:
        print("\n🚀 DRY_RUN is OFF. Phase 4 will CREATE and PUBLISH these agents.")


display_agent_plan(agent_plan)

#### Optional: Edit the plan manually

Uncomment and modify the cell below to adjust agent names, merge/split agents, or change instructions before creation.

In [ ]:
# # Example: Rename an agent
# agent_plan["agents"][0]["name"] = "Custom Fraud & Payment Agent"
#
# # Example: Merge agents 0 and 1
# merged = agent_plan["agents"][0]
# merged["data_sources"] += agent_plan["agents"][1]["data_sources"]
# merged["name"] = "Merged Agent"
# agent_plan["agents"] = [merged] + agent_plan["agents"][2:]
#
# # Example: Remove an agent
# agent_plan["agents"] = [a for a in agent_plan["agents"] if a["name"] != "Unwanted Agent"]
#
# # Re-validate after edits
# agent_plan = split_oversized_agents(agent_plan)
# display_agent_plan(agent_plan)

---
## Phase 4 — Data Agent Creation & Configuration

Creates, configures, and publishes Data Agents using the `fabric-data-agent-sdk`.

In [ ]:
# Map item_type strings to SDK datasource type parameter
ITEM_TYPE_TO_SDK_TYPE = {
    "Lakehouse": "lakehouse",
    "Warehouse": "warehouse",
    "SemanticModel": "semanticmodel",
    "KQLDatabase": "kqldatabase",
}


def create_agents_from_plan(plan: dict, dry_run: bool) -> list[dict]:
    """Create and configure Data Agents based on the validated plan."""
    created_agents = []
    agents = plan.get("agents", [])
    
    for i, agent_def in enumerate(agents, 1):
        agent_name = f"{AGENT_NAME_PREFIX} - {agent_def['name']}"
        # Sanitize name (remove special chars that might cause issues)
        agent_name = re.sub(r'[^\w\s\-]', '', agent_name).strip()[:256]
        
        print(f"\n{'='*60}")
        print(f"[{i}/{len(agents)}] Agent: {agent_name}")
        print(f"{'='*60}")
        
        if dry_run:
            print("  [DRY RUN] Would create agent. Skipping.")
            created_agents.append({
                "agent_name": agent_name,
                "status": "dry_run",
                "data_sources": len(agent_def.get("data_sources", [])),
                "description": agent_def.get("description", ""),
            })
            continue
        
        # Step 1: Create the agent
        try:
            print(f"  Creating agent '{agent_name}'...")
            data_agent = create_data_agent(agent_name)
            print(f"  ✓ Agent created.")
        except Exception as e:
            print(f"  ✗ Failed to create agent: {e}")
            created_agents.append({"agent_name": agent_name, "status": f"create_failed: {e}"})
            continue
        
        # Step 2: Set system-level instructions
        try:
            sys_instructions = agent_def.get("system_instructions", "")
            if sys_instructions:
                # Truncate to 15,000 chars (SDK limit)
                data_agent.update_configuration(instructions=sys_instructions[:15000])
                print(f"  ✓ System instructions set ({len(sys_instructions)} chars).")
        except Exception as e:
            print(f"  ⚠ Failed to set system instructions: {e}")
        
        # Step 3: Add data sources and configure each
        for ds_def in agent_def.get("data_sources", []):
            ds_item_name = ds_def["item_name"]
            ds_item_type = ds_def["item_type"]
            sdk_type = ITEM_TYPE_TO_SDK_TYPE.get(ds_item_type, ds_item_type.lower())
            
            try:
                print(f"  Adding datasource '{ds_item_name}' ({sdk_type})...")
                data_agent.add_datasource(ds_item_name, type=sdk_type)
                print(f"  ✓ Datasource added.")
            except Exception as e:
                print(f"  ✗ Failed to add datasource '{ds_item_name}': {e}")
                continue
            
            # Get the datasource object
            try:
                datasources = data_agent.get_datasources()
                datasource = None
                for ds_obj in datasources:
                    # Match by name — the SDK doesn't expose a direct lookup
                    if hasattr(ds_obj, 'name') and ds_obj.name == ds_item_name:
                        datasource = ds_obj
                        break
                if datasource is None:
                    # Fallback: take the last added datasource
                    datasource = datasources[-1]
            except Exception as e:
                print(f"  ⚠ Cannot retrieve datasource object: {e}")
                continue
            
            # Step 3a: Select tables
            selected_tables = ds_def.get("selected_tables", [])
            for table_name in selected_tables:
                try:
                    datasource.select("dbo", table_name)
                except Exception:
                    # Try without schema prefix
                    try:
                        datasource.select("", table_name)
                    except Exception as e:
                        print(f"    ⚠ Cannot select table '{table_name}': {e}")
            if selected_tables:
                print(f"    ✓ Selected {len(selected_tables)} tables.")
            
            # Step 3b: Set per-datasource instructions
            ds_instructions = ds_def.get("data_instructions", "")
            if ds_instructions:
                try:
                    datasource.update_configuration(instructions=ds_instructions[:15000])
                    print(f"    ✓ Data instructions set.")
                except Exception as e:
                    print(f"    ⚠ Failed to set data instructions: {e}")
            
            # Step 3c: Add example queries (not for Semantic Models)
            example_queries = ds_def.get("example_queries", [])
            if example_queries and ds_item_type != "SemanticModel":
                fewshot_dict = {}
                for eq in example_queries[:MAX_EXAMPLES_PER_DATASOURCE]:
                    q = eq.get("question", "")
                    sql = eq.get("sql", "")
                    if q and sql:
                        fewshot_dict[q] = sql
                
                if fewshot_dict:
                    try:
                        datasource.add_fewshots(fewshot_dict)
                        print(f"    ✓ Added {len(fewshot_dict)} example queries.")
                    except Exception as e:
                        print(f"    ⚠ Failed to add example queries: {e}")
        
        # Step 4: Publish
        try:
            data_agent.publish()
            print(f"  ✓ Agent published!")
            status = "published"
        except Exception as e:
            print(f"  ⚠ Failed to publish agent: {e}")
            status = "created_not_published"
        
        created_agents.append({
            "agent_name": agent_name,
            "status": status,
            "data_sources": len(agent_def.get("data_sources", [])),
            "description": agent_def.get("description", ""),
        })
        
        # Small delay to avoid rate limits
        time.sleep(2)
    
    return created_agents


# Execute
print(f"DRY_RUN = {DRY_RUN}")
created_agents = create_agents_from_plan(agent_plan, DRY_RUN)

---
## Phase 5 — Report & Agent Garden Catalog

Generates a summary report and optionally writes the catalog to a Delta table.

In [ ]:
# Build catalog DataFrame
catalog_records = []
for agent_def, result in zip(agent_plan.get("agents", []), created_agents):
    ds_names = [ds["item_name"] for ds in agent_def.get("data_sources", [])]
    ds_types = [ds["item_type"] for ds in agent_def.get("data_sources", [])]
    all_tables = []
    for ds in agent_def.get("data_sources", []):
        all_tables.extend(ds.get("selected_tables", []))
    
    catalog_records.append({
        "agent_name": result["agent_name"],
        "description": agent_def.get("description", ""),
        "status": result["status"],
        "data_source_count": len(ds_names),
        "data_sources": ", ".join(ds_names),
        "data_source_types": ", ".join(ds_types),
        "tables": ", ".join(all_tables),
        "table_count": len(all_tables),
        "system_instructions_length": len(agent_def.get("system_instructions", "")),
        "total_example_queries": sum(
            len(ds.get("example_queries", [])) for ds in agent_def.get("data_sources", [])
        ),
    })

catalog_df = pd.DataFrame(catalog_records)

print("\n" + "=" * 80)
print("AGENT GARDEN CATALOG — Summary")
print("=" * 80)
display(catalog_df)

print(f"\nTotal agents: {len(catalog_df)}")
print(f"Total data sources used: {catalog_df['data_source_count'].sum()}")
print(f"Total tables mapped: {catalog_df['table_count'].sum()}")
print(f"Total example queries: {catalog_df['total_example_queries'].sum()}")

In [ ]:
# Write catalog to Delta table (optional)
if CATALOG_LAKEHOUSE and not DRY_RUN:
    print(f"Writing catalog to '{CATALOG_LAKEHOUSE}.{CATALOG_TABLE_NAME}'...")
    
    spark_df = spark.createDataFrame(catalog_df)
    spark_df.write.mode("overwrite").saveAsTable(f"{CATALOG_LAKEHOUSE}.{CATALOG_TABLE_NAME}")
    
    print(f"✓ Catalog written to Delta table '{CATALOG_LAKEHOUSE}.{CATALOG_TABLE_NAME}'.")
    print("  You can now build a Power BI report on this table for the Agent Garden UI.")
else:
    if DRY_RUN:
        print("Catalog write skipped (DRY_RUN mode).")
    elif not CATALOG_LAKEHOUSE:
        print("Catalog write skipped (CATALOG_LAKEHOUSE not configured).")

In [ ]:
# Save the full agent plan as JSON artifact for reference
plan_json = json.dumps(agent_plan, indent=2, ensure_ascii=False)
print("Full agent plan JSON (for reference):")
print(plan_json[:5000])
if len(plan_json) > 5000:
    print(f"\n... ({len(plan_json):,} total characters. Full plan available in 'agent_plan' variable.)")

---
## Done!

### Next Steps:

1. **Review dry-run output** above — check that agent groupings and instructions make sense
2. **Set `DRY_RUN = False`** in Cell 0 and re-run from Phase 4 to actually create the agents
3. **Test agents** by navigating to them in the Fabric workspace and asking questions
4. **Share agents** via the Copilot in Power BI integration or published URLs
5. **Build Agent Garden UI** by creating a Power BI report on the `agent_garden_catalog` Delta table

### Troubleshooting:

- **"Cannot list items"**: Ensure you have Viewer (or higher) role on the target workspaces
- **"Cannot profile table"**: The Spark context must have access to the lakehouse data (may need shortcuts)
- **"Failed to add datasource"**: The item must be accessible via the OneLake catalog from the notebook's workspace
- **Azure OpenAI errors**: Verify `AOAI_ENDPOINT` and `AOAI_DEPLOYMENT` are correct and that your token has access
- **Max 5 data sources**: The notebook auto-splits agents that exceed this limit